In [0]:
# CELL 1: Install & Verify Libraries
# -------------------------------------------------------------------------
# 1. Install the libraries
%pip install beautifulsoup4 gdown
 
# 2. Verify installation immediately
try:    
    from bs4 import BeautifulSoup
    import gdown
    print("✅ BeautifulSoup4 and gdown are installed and working correctly.")
except ImportError as e:
    print(f"❌ Error: {e}. Please re-run this cell.")

In [0]:
# CELL 2: Setup, Directory Creation & File Download
# -------------------------------------------------------------------------
import gdown
import os
import tempfile
import shutil
from datetime import datetime, timedelta
from azure.storage.filedatalake import DataLakeServiceClient
 
# -------------------------------------------------------------------------
# 1. Google Drive Source (direct file link)
# -------------------------------------------------------------------------
gdrive_file_id = "1nB-wGjdlH8YcEGJcFMGdRNpbI1bEErt6"
 
# -------------------------------------------------------------------------
# 2. Azure Storage Configuration
# -------------------------------------------------------------------------
storage_account_name = "databrickslabjp"
storage_account_key = "REMOVED_FOR_SECURITY"
 
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")
 
# Define paths
landing_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/landing/"
archive_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/archive/"
 
# -------------------------------------------------------------------------
# 3. Create Directory Structure (if it doesn't exist)
# -------------------------------------------------------------------------
dbutils.fs.mkdirs(landing_zone)
dbutils.fs.mkdirs(archive_zone)
print("✅ Directory structure created/verified:")
print(f"   Landing: {landing_zone}")
print(f"   Archive: {archive_zone}")
 
# -------------------------------------------------------------------------
# 4. Download transactions.csv & Upload to ADLS
# -------------------------------------------------------------------------
def download_and_ingest():
    """Download transactions.csv from Google Drive and stream-upload to ADLS."""
    tmp_dir = tempfile.mkdtemp()
    local_path = os.path.join(tmp_dir, "transactions.csv")
 
    try:
        # Download single file using its Google Drive file ID
        print(f"\n🔍 Downloading transactions.csv from Google Drive...")
        gdown.download(id=gdrive_file_id, output=local_path, quiet=False)
 
        file_size = os.path.getsize(local_path)
        print(f"✅ Downloaded: transactions.csv ({file_size:,} bytes)")
 
        # Stream-upload to ADLS using Azure SDK (memory-efficient)
        service = DataLakeServiceClient(
            account_url=f"https://{storage_account_name}.dfs.core.windows.net",
            credential=storage_account_key
        )
        fs_client = service.get_file_system_client("bronze")
        file_client = fs_client.get_file_client("transactions/landing/transactions.csv")
 
        print(f"⬆️  Uploading to ADLS landing zone...")
        with open(local_path, "rb") as data:
            file_client.upload_data(data, overwrite=True)
 
        dest_path = landing_zone + "transactions.csv"
        print(f"✅ Uploaded to: {dest_path}")
        return dest_path
 
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
 
downloaded_path = download_and_ingest()
print(f"\n🎉 Configuration Loaded & File Ingested Successfully!")

In [0]:
# CELL 3: Retention Policy
# -------------------------------------------------------------------------
def cleanup_old_files(days_to_keep=14):
    print("🧹 Checking archive for old files...")
    cutoff_date = datetime.now() - timedelta(days=days_to_keep)
 
    try:
        files = dbutils.fs.ls(archive_zone)
        for f in files:
            # Expected format: YYYY-MM-DD.csv
            filename = f.name.replace(".csv", "")
            try:
                file_date = datetime.strptime(filename, "%Y-%m-%d")
                if file_date < cutoff_date:
                    print(f" 🗑️ Deleting: {f.name}")
                    dbutils.fs.rm(f.path)
            except ValueError:
                continue
    except Exception:
        print(" ℹ️ Archive is empty or does not exist yet.")
 
cleanup_old_files()

In [0]:
# CELL 4: Archival Logic
# -------------------------------------------------------------------------
def archive_current_file():
    current_file_path = f"{landing_zone}current.csv"
 
    try:
        # Check if file exists
        dbutils.fs.ls(current_file_path)
 
        # Rename to YYYY-MM-DD.csv
        today_str = datetime.now().strftime("%Y-%m-%d")
        archive_path = f"{archive_zone}{today_str}.csv"
 
        print(f"📦 Archiving previous 'current.csv' to: {archive_path}")
        dbutils.fs.mv(current_file_path, archive_path)
 
    except Exception:
        print("ℹ️ No 'current.csv' found in landing. First run?")
 
archive_current_file()

In [0]:
# CELL 5: Download Snapshot (Robust Version with Retries)
# -------------------------------------------------------------------------
import os
import time
 
def download_snapshot_with_retry(max_retries=3):
    fresh_url = get_dynamic_download_url()
    local_path = "/tmp/current.csv"
    headers = {'User-Agent': 'Mozilla/5.0'}
 
    for attempt in range(1, max_retries + 1):
        try:
            print(f"⬇️ Attempt {attempt}/{max_retries}: Downloading to driver node...")
 
# 30-second timeout, Stream=True
            with requests.get(fresh_url, headers=headers, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(local_path, 'wb') as f:
# 10MB Chunks
                    for chunk in r.iter_content(chunk_size=10*1024*1024):
                        if chunk:
                            f.write(chunk)
 
            print(" ✅ Download complete.")
            break # Exit loop on success
 
        except Exception as e:
            print(f" ⚠️ Attempt {attempt} failed: {str(e)}")
            if attempt < max_retries:
                print(" ⏳ Waiting 5 seconds...")
                time.sleep(5)
            else:
                print("❌ All retries failed.")
            raise e
 
# Upload to Data Lake
    if os.path.exists(local_path):
        print(f"⬆️ Uploading to Bronze Landing Zone...")
        dbutils.fs.cp(f"file:{local_path}", f"{landing_zone}current.csv")
        os.remove(local_path)
        print("✅ Success! 'current.csv' is updated in the Data Lake.")
 
# Run it
    download_snapshot_with_retry()